# 0 — Background & Theory: Univariate Time Series (AR, MA, ARMA, ARIMA, SARIMA) and Model Evaluation

**Companion notebook to:** `1_arma.ipynb`, `2_arima.ipynb`, `3_more_on_model_evaluation.ipynb`, and *Huy H., "Building Statistical Models in Python", Chapter 11 — ARIMA Models*.

**Purpose of this notebook:** give you the theory in one place, explain *why* each step of the modeling workflow exists, and be explicit about what this class of model can and cannot do — before you touch the lab notebooks.

**How to use the full set of materials in this bundle:**

| File | Purpose |
|---|---|
| `00_Background_and_Theory.ipynb` | This file. Read first. |
| `01_Extended_Lab.ipynb` | The lab handout — tasks, questions, no filled-in code. |
| `02_Skeleton_Practice.ipynb` | The same lab, structured as a notebook with `# TODO` stubs, so you can code it from scratch. |
| `03_Cheat_Sheet.ipynb` | One-page-per-topic quick reference of runnable snippets (identification rules, tests, model fitting, diagnostics). |
| `04_Reusable_Project_Template.ipynb` | A generic, parameterized pipeline you can point at *any* univariate series for a new project. |
| `05_Solutions.ipynb` | Fully worked solutions to `01`/`02`, with commentary. |


## 1. The big picture: what problem are we solving?

We have a single sequence of numbers ordered in time, $y_1, y_2, \dots, y_T$ (a stock price, GDP, passenger counts). We want to:

1. **Describe** the dependence structure — how does $y_t$ relate to its own past?
2. **Fit** a compact statistical model that captures that structure.
3. **Forecast** future values with a defensible measure of uncertainty.
4. **Evaluate** honestly whether the model actually generalizes, not just fits historical noise.

Everything in this chapter is a special case of one idea: *if a series is stationary, its past linearly explains a good chunk of its future; if it isn't stationary, transform it (differencing) until it is, model the transformed series, then invert the transform to forecast the original.*


## 2. Stationarity — the load-bearing assumption

A process is **(weakly/covariance) stationary** if:
- $E[y_t] = \mu$ is constant over time,
- $\mathrm{Var}(y_t) = \sigma^2$ is constant over time,
- $\mathrm{Cov}(y_t, y_{t+k})$ depends only on the lag $k$, not on $t$.

Why we care: **every AR/MA/ARMA formula below (autocorrelation formulas, the AIC/BIC order-selection routine, confidence intervals from `statsmodels`) is derived under stationarity.** Fit an ARMA model to a non-stationary series and the coefficients, standard errors, and forecasts are not trustworthy — even if the software returns numbers without complaint.

**Root condition (equivalent, more mechanical way to check stationarity for AR):** write the AR polynomial in the backshift operator $B$ ($By_t = y_{t-1}$):

$$ (1-\phi_1 B - \phi_2 B^2 - \dots - \phi_p B^p)\,y_t = \epsilon_t $$

Solve $1-\phi_1 z - \dots - \phi_p z^p = 0$ for the roots $z$. The process is stationary iff **all roots lie outside the unit circle**, i.e. $|z| > 1$ for every root (real roots: $|z|>1$ directly; complex conjugate roots $a\pm bi$: check $\sqrt{a^2+b^2} > 1$).

- **AR(1)** intuition: $y_t = \phi_1 y_{t-1} + \epsilon_t$ is stationary iff $|\phi_1| < 1$. The root is $z = 1/\phi_1$.
- If $|\phi_1| = 1$ → **unit root** → non-stationary but *differencing* removes it (this is exactly the "I" in ARIMA).
- If $|\phi_1| > 1$ → **explosive** → cannot be sensibly modeled at all (not even by differencing).

**Practical, data-driven test: the Augmented Dickey-Fuller (ADF) test.**
- $H_0$: a unit root is present (series is non-stationary / I(1)).
- $H_1$: no unit root (series is stationary / I(0)).
- Small p-value (e.g. < 0.05) → reject $H_0$ → treat as stationary.
- The ADF test has **low power** (prone to Type II error), so practitioners usually use a generous `maxlag` and corroborate with a visual read of the ACF (autocorrelations that decay very slowly = trend/unit-root signature).


## 3. Invertibility — the MA-side analogue of stationarity

For a moving-average process
$$ y_t = (1-\theta_1 B - \dots - \theta_q B^q)\,\epsilon_t $$
we require **invertibility**: all roots of $1-\theta_1 z - \dots - \theta_q z^q = 0$ must lie outside the unit circle. Invertibility guarantees the process can be re-expressed as an (infinite-order, converging) AR process, which is what lets us recover a unique, sensible relationship between the observed series and its own past errors. An MA(1) is invertible iff $|\theta_1| < 1$.

If a fitted MA/ARMA/ARIMA model comes back non-invertible, `statsmodels` will warn you — this almost always means the order is wrong or the series still has a unit root that needs differencing/seasonal differencing first.


## 4. Model family cheat-map

| Model | Equation (zero-mean, backshift form) | What it captures | Identify order from |
|---|---|---|---|
| **AR(p)** | $(1-\phi_1B-\dots-\phi_pB^p)y_t=\epsilon_t$ | $y_t$ is a direct linear function of its own last $p$ values | **PACF** — cuts off after lag $p$ |
| **MA(q)** | $y_t=(1-\theta_1B-\dots-\theta_qB^q)\epsilon_t$ | $y_t$ is a function of the last $q$ *shocks* (a low-pass filter on noise) | **ACF** — cuts off after lag $q$ |
| **ARMA(p,q)** | $\Phi(B)y_t=\Theta(B)\epsilon_t$ | both effects at once; more parsimonious than a pure high-order AR or MA | Both ACF & PACF **taper/dampen** (neither cuts off cleanly) |
| **ARIMA(p,d,q)** | $\Phi(B)(1-B)^d y_t = \Theta(B)\epsilon_t$ | ARMA applied to the $d$-times-differenced series, to handle a trend/unit root | ADF test → choose $d$; then ACF/PACF of the differenced series → choose $p,q$ |
| **ARIMA(p,d,q)(0,D,0)[s]** (seasonal differencing only, as used in the book) | adds $(1-B^s)^D$ | removes a repeating pattern with period $s$ before modeling | peak in ACF at lag $s$ → seasonal period; ADF on seasonally-differenced series → $D$ |
| **Full SARIMA(p,d,q)(P,D,Q)[s]** | adds seasonal AR/MA polynomials too | seasonal *and* non-seasonal autocorrelation, e.g. "this January is like every other January, with decaying influence from recent Januaries" | `pmdarima.auto_arima(..., seasonal=True, m=s)` search; beyond the scope of the source chapter but mentioned for context |

**Key identification heuristic** (worth memorizing): *AR cuts off in PACF and decays in ACF. MA cuts off in ACF and decays in PACF. ARMA decays in both.* This works because an invertible finite MA is equivalent to an infinite (decaying) AR, and a stationary finite AR is equivalent to an infinite (decaying) MA.


## 5. The end-to-end workflow used throughout the lab

1. **Visual inspection** — plot the realization, ACF, and PACF (`statsmodels.graphics.tsaplots.plot_acf/plot_pacf`).
2. **Trend/unit-root check** — Augmented Dickey-Fuller test (`statsmodels.tsa.stattools.adfuller` or `pmdarima.arima.ADFTest`). Choose a `maxlag` at least as large as the farthest lag with a visually significant ACF spike, since a strong, slowly-decaying trend can hide in later lags.
3. **Seasonality check** (if relevant) — look for a periodic spike in the ACF at lag $s$; remove it first via seasonal differencing $y_t - y_{t-s}$, *then* re-check for a residual trend on the seasonally-differenced series.
4. **Order selection** — `statsmodels.tsa.stattools.arma_order_select_ic` (grid search over $(p,q)$ scored by AIC/BIC) or `pmdarima.auto_arima` (stepwise search, also handles $d$ and seasonal orders automatically). AIC and BIC frequently disagree — BIC penalizes extra parameters more heavily and tends to pick the more parsimonious (lower order) model; when they disagree, fit and compare both.
5. **Fit** — `statsmodels.tsa.arima.model.ARIMA(y, order=(p,d,q), enforce_stationarity=True, enforce_invertibility=True).fit()`.
6. **Diagnose** — read the coefficient table (drop/reconsider terms whose 95% CI contains 0), and the four built-in tests in `.summary()`:
   - **Ljung-Box (Prob(Q))** — high p-value ⇒ no leftover autocorrelation in residuals (good; residuals look like white noise).
   - **Jarque-Bera (Prob(JB))** — high p-value ⇒ residuals consistent with normality.
   - **Heteroskedasticity (Prob(H))** — high p-value ⇒ residual variance looks constant over time.
   - **Skew / Kurtosis** — skew in $[-0.5,0.5]$ = not skewed; kurtosis near 3 = normal-like tails.
7. **Backtest** — hold out the last $h$ points, fit on the rest, forecast $h$ steps, compare to actuals (MSE/RMSE). This is the single most important step for catching an overfit model that only looks good on AIC/BIC.
8. **Forecast** — extend `.get_prediction(start=len(y), end=len(y)+h-1)` beyond the observed data, always reporting the confidence interval, not just the point forecast.


## 6. Model evaluation techniques beyond AIC/BIC (Chapter 3 of the lab set)

AIC/BIC only tell you about **in-sample fit penalized for complexity** — they do not prove a model will forecast well. The lab's model-evaluation notebook introduces four complementary, more empirical techniques:

- **Resampling** (`.resample('M').mean()`) — change the time granularity (e.g. daily → monthly) to smooth noise, match business reporting cadence, or make a series more tractable to model. Downsampling trades resolution for stability.
- **Shifting** (`.shift(k)`) — create lagged copies of a series as engineered features, or to visually/statistically compare leading vs. lagging relationships (cross-correlation).
- **Optimized persistence forecasting** — the naive baseline "$\hat y_{t+1} = y_{t+1-p}$" (repeat the value from $p$ steps ago) swept over a range of $p$, scored by RMSE on a held-out window. **Any real model should beat the best persistence baseline** — if it doesn't, the extra modeling machinery isn't earning its keep.
- **Rolling-window forecasting** — "$\hat y_{t+1} = \text{mean}(y_{t-w+1},\dots,y_t)$" swept over window size $w$, again scored by RMSE. A slightly smarter, still model-free baseline.

Both baselines matter because they're computationally trivial and assumption-free — they define the bar an ARIMA/SARIMA model must clear to be worth the extra complexity.


## 7. Limitations of this model family — read before you trust a forecast

Be explicit with yourself (and any stakeholder) about what AR/MA/ARMA/ARIMA/SARIMA **cannot** do:

1. **Linearity.** These are linear models in the lagged values and lagged errors. Non-linear dynamics (regime switching, threshold effects, volatility clustering) are invisible to them.
2. **Constant variance assumption.** ARIMA models a constant-variance process. Financial series like the Coca-Cola stock price in this lab exhibit **volatility clustering** (calm periods followed by turbulent ones) — the Heteroskedasticity test in the model summary is exactly the diagnostic that catches this violation. When it's violated, prediction intervals are systematically wrong (usually too narrow during turbulent periods). Modeling changing variance requires a different model family (e.g., GARCH), which is out of scope here.
3. **Univariate.** No exogenous information — no interest rates, competitor prices, macro shocks, earnings announcements, holidays (beyond fixed seasonal patterns) can enter the model. `SARIMAX` supports exogenous regressors but that's a different, larger modeling exercise (also flagged as "the next chapter" in the source material).
4. **Short-horizon reliability, long-horizon fragility.** Confidence intervals widen quickly; forecasts many steps ahead regress toward the unconditional trend/mean and lose the ability to track real turning points. The book's own AR(4)/ARMA examples and the GDP/airline ARIMA forecasts are only validated a handful of steps ahead.
5. **Structural breaks are invisible until they're in the training window.** A model fit on 2016-2019 data has no way to "know" about a 2020-style shock; it will confidently extrapolate the old pattern (see the Coca-Cola 2020 dip in the model-evaluation notebook — persistence/rolling baselines get this visibly wrong too, illustrating that *no* purely time-based model handles unforeseen shocks).
6. **Near-random-walk series are (correctly) unforecastable beyond very short horizons.** Efficient-market theory implies daily stock **prices** are close to a random walk; realistic ARIMA fits on price levels often reduce to something like ARIMA(0,1,0) (a random walk with drift) — i.e., the "best" model honestly says "tomorrow ≈ today," not "we found a secret pattern." Treat any ARIMA model that claims strong predictive power on raw daily stock prices with real skepticism; it's more likely overfit to noise than to have found genuine structure.
7. **Small samples / high order = overfitting risk.** The lab explicitly demonstrates this: an ARMA(4,1) fit had insignificant, near-zero AR coefficients (an overfit artifact of chasing the lowest AIC) while the simpler, all-significant ARMA(2,1) was the more defensible, generalizable model. Lower training error is not evidence of a better model — always sanity-check coefficient significance and out-of-sample performance, not just AIC/BIC.
8. **The ADF test's low power** means you can fail to detect a genuine unit root, especially with a `maxlag` that's too small; corroborate statistical tests with visual inspection of the ACF/realization plot.


## 8. What this class of lab *can* be used to simulate/demonstrate successfully

To calibrate expectations in the other direction — this is genuinely useful, well-scoped territory:

- **Teaching/simulation with known ground truth.** Because `statsmodels.tsa.ArmaProcess` lets you *generate* data from a process whose true $(\phi, \theta)$ you specify, you can verify that the identification workflow (ACF/PACF → AIC/BIC → fitted coefficients) recovers something close to the truth. This is the single best use of this material: build intuition by round-tripping "simulate → forget the truth → re-derive it → compare."
- **Smooth, strongly seasonal series with a stable underlying pattern** (classic case: the airline-passengers dataset) — SARIMA-style seasonal differencing captures this kind of series very well, and short-horizon forecasts track the actual data closely (see Figure 11.25-equivalent forecast in this lab).
- **Slow-moving macro aggregates** (GDP-like series) — a single, well-chosen difference plus a low-order ARMA fit is often "good enough" for short-horizon forecasting, precisely because such series change gradually and rarely have abrupt regime shifts within a short forecast window.
- **Establishing a rigorous baseline.** Even where ARIMA/SARIMA isn't the final production model, the identification + backtesting + persistence/rolling-baseline workflow in this lab is exactly the right first step before reaching for something more complex (SARIMAX, state-space models, ML-based forecasters, or GARCH-family volatility models).
- **Diagnosing *why* a series is hard to forecast.** Even a "failed" ARIMA fit (e.g., collapsing to a random walk on daily stock prices) is informative — it correctly tells you the series has little linear autocorrelation structure left to exploit, which is a genuine and useful finding, not a failure of the exercise.

**Bottom line:** use this toolkit to (a) build a rigorous, well-diagnosed baseline forecast, and (b) understand precisely how much of a series' behavior is linearly predictable from its own past — not as a tool for confidently timing volatile markets or handling structural breaks.
